# AI Cup 2026 — Bird Radar Track Classification
Trains a LightGBM model with group-aware CV, generates `submission.csv`.

## Model Hyperparameters

In [ ]:
# ─── Model Hyperparameters (update after grid search) ───
N_ESTIMATORS      = 1000
LEARNING_RATE     = 0.05
NUM_LEAVES        = 63
MIN_CHILD_SAMPLES = 10
SUBSAMPLE         = 0.8
COLSAMPLE_BYTREE  = 0.8
CLASS_WEIGHT      = 'balanced'
DEVICE            = 'gpu'

# ─── CV Settings ───
N_SPLITS          = 10
RANDOM_STATE      = 42

## Imports

In [ ]:
import numpy as np
import pandas as pd
from shapely import wkb
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier
import lightgbm as lgb
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek

## 1. Load Data (KNMI-enriched)

In [ ]:
print("Loading KNMI-enriched datasets...")
train_df = pd.read_csv("dataset/train_with_knmi_286.csv").set_index("track_id")
test_df = pd.read_csv("dataset/test_with_knmi_286.csv").set_index("track_id")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

## 2. Trajectory Parsing & Feature Extraction

In [ ]:
def parse_trajectory(hex_str):
    """Decode EWKB hex string into a list of (lon, lat, alt, rcs) tuples."""
    if not isinstance(hex_str, str) or len(hex_str) == 0:
        return []
    try:
        geom = wkb.loads(bytes.fromhex(hex_str) if not hex_str.startswith('\\x01') else hex_str, hex=True)
        if geom.geom_type == 'LineString':
            return list(geom.coords)
        elif geom.geom_type == 'Point':
            return [geom.coords[0]]
        elif geom.geom_type in ('MultiPoint', 'GeometryCollection'):
            return [g.coords[0] for g in geom.geoms]
    except Exception:
        pass
    return []


def trajectory_features(row):
    """Extract spatial, RCS, and velocity features from a trajectory."""
    coords = parse_trajectory(row['trajectory'])
    n = len(coords)
    if n == 0:
        return pd.Series({})

    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    alts = [c[2] for c in coords] if len(coords[0]) > 2 else [np.nan] * n
    rcs  = [c[3] for c in coords] if len(coords[0]) > 3 else [np.nan] * n

    # Horizontal displacement (degrees -> approx metres at ~53 deg N)
    dx = np.diff(lons) * 71000
    dy = np.diff(lats) * 111000
    step_dist = np.sqrt(dx**2 + dy**2)
    total_dist = step_dist.sum() if len(step_dist) > 0 else 0.0

    # Tortuosity (turning behaviour)
    bearings = np.arctan2(dy, dx)
    bearing_changes = np.abs(np.diff(bearings)) if len(bearings) > 1 else np.array([0.0])
    bearing_changes = np.minimum(bearing_changes, 2 * np.pi - bearing_changes)

    feats = {
        'n_points':       n,
        'total_dist_m':   total_dist,
        'mean_step_m':    step_dist.mean() if len(step_dist) > 0 else 0.0,
        'std_step_m':     step_dist.std()  if len(step_dist) > 0 else 0.0,
        'lon_range':      max(lons) - min(lons),
        'lat_range':      max(lats) - min(lats),
        'alt_mean':       np.nanmean(alts),
        'alt_std':        np.nanstd(alts),
        'rcs_mean':       np.nanmean(rcs),
        'rcs_std':        np.nanstd(rcs),
        'rcs_min':        np.nanmin(rcs),
        'rcs_max':        np.nanmax(rcs),
        'tortuosity':     bearing_changes.mean() if len(bearing_changes) > 0 else 0.0,
        'tortuosity_max': bearing_changes.max()  if len(bearing_changes) > 0 else 0.0,
    }

    # Speed & acceleration from trajectory_time
    times = row.get('trajectory_time', '')
    t_list = []
    if isinstance(times, str) and times.strip():
        try:
            t_list = [float(x) for x in times.strip('[]').split(',')]
        except ValueError:
            pass
    elif isinstance(times, (list, np.ndarray)):
        t_list = list(times)

    if len(t_list) == n and n > 1:
        dt = np.diff(t_list)
        dt = np.where(dt == 0, 1e-6, dt)  # avoid div-by-zero
        speeds = step_dist / dt
        feats['speed_mean'] = np.mean(speeds)
        feats['speed_std']  = np.std(speeds)
        feats['speed_max']  = np.max(speeds)

        if len(speeds) > 1:
            accel = np.diff(speeds) / dt[1:]
            feats['accel_mean'] = np.mean(accel)
            feats['accel_std']  = np.std(accel)
        else:
            feats['accel_mean'] = 0.0
            feats['accel_std']  = 0.0
    else:
        feats['speed_mean'] = np.nan
        feats['speed_std']  = np.nan
        feats['speed_max']  = np.nan
        feats['accel_mean'] = np.nan
        feats['accel_std']  = np.nan

    return pd.Series(feats)

## 3. Feature Engineering

In [ ]:
for df in [train_df, test_df]:
    # Timestamps
    df['ts_start'] = pd.to_datetime(df['timestamp_start_radar_utc'], utc=True)
    df['ts_end']   = pd.to_datetime(df['timestamp_end_radar_utc'],   utc=True)
    df['duration_s'] = (df['ts_end'] - df['ts_start']).dt.total_seconds()

    # Time-of-day / seasonality
    df['hour']  = df['ts_start'].dt.hour
    df['month'] = df['ts_start'].dt.month
    df['is_daytime'] = ((df['hour'] >= 6) & (df['hour'] <= 20)).astype(int)

    # Cyclical encoding
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Derived radar features
    df['alt_range']      = df['max_z'] - df['min_z']
    df['airspeed_per_m'] = df['airspeed'] / (df['max_z'] + 1)

# Trajectory features
print("Extracting trajectory features for train_df...")
train_df = train_df.join(train_df.apply(trajectory_features, axis=1))
print("Extracting trajectory features for test_df...")
test_df = test_df.join(test_df.apply(trajectory_features, axis=1))

## 4. Feature List

In [ ]:
base_features = [
    'airspeed', 'min_z', 'max_z', 'duration_s', 'radar_bird_size',
    'hour', 'month', 'is_daytime',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'alt_range', 'airspeed_per_m',
]

trajectory_feats = [
    'n_points', 'total_dist_m', 'mean_step_m', 'std_step_m',
    'lon_range', 'lat_range', 'alt_mean', 'alt_std',
    'rcs_mean', 'rcs_std', 'rcs_min', 'rcs_max',
    'tortuosity', 'tortuosity_max',
    'speed_mean', 'speed_std', 'speed_max',
    'accel_mean', 'accel_std',
]

knmi_features = [
    'knmi_286_wind_direction_degrees',
    'knmi_286_hourly_mean_wind_speed_mps',
    'knmi_286_wind_speed_at_observation_mps',
    'knmi_286_max_wind_gust_mps',
    'knmi_286_air_temperature_c',
    'knmi_286_dew_point_temperature_c',
    'knmi_286_sunshine_duration_hours',
    'knmi_286_global_radiation_j_cm2',
    'knmi_286_precipitation_duration_hours',
    'knmi_286_precipitation_amount_mm',
    'knmi_286_relative_humidity_percent',
    'knmi_286_weather_indicator_code',
    'knmi_286_wind_dir_sin',
    'knmi_286_wind_dir_cos',
    'knmi_286_wind_dir_variable',
]

features = base_features + trajectory_feats + knmi_features

X = train_df[features]
X_test = test_df[features]
y = train_df['bird_group']

print(f"Feature matrix: X={X.shape}, X_test={X_test.shape}, classes={y.nunique()}")

## 5. Model & Pipeline

In [ ]:
numeric_features = [f for f in features if f != 'radar_bird_size']
categorical_features = ['radar_bird_size']
cat_indices = [len(numeric_features) + i for i in range(len(categorical_features))]

imputer = ColumnTransformer([
    ('num_imputer', SimpleImputer(strategy='median'), numeric_features),
    ('cat_imputer', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), categorical_features)
])

smotenc_tomek = SMOTETomek(
    smote=SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE
)

lgb_model = LGBMClassifier(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    num_leaves=NUM_LEAVES,
    min_child_samples=MIN_CHILD_SAMPLES,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    device=DEVICE,
    verbose=-1,
)

pipeline = ImbPipeline([
    ('imputer', imputer),
    ('oversampler', smotenc_tomek),
    ('model', lgb_model)
])

## 6. Group-Aware Cross-Validation + Training

In [ ]:
groups = train_df['primary_observation_id']
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
split = list(cv.split(X, y, groups))

classes = np.sort(y.unique())
oof_preds = pd.DataFrame(0.0, index=X.index, columns=classes)
test_preds = np.zeros((len(X_test), len(classes)))

print(f"Training {N_SPLITS}-fold StratifiedGroupKFold...")
for i, (train_idx, val_idx) in enumerate(split):
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold   = X.iloc[val_idx]
    y_val_fold   = y.iloc[val_idx]

    pipeline_fold = clone(pipeline)
    pipeline_fold.fit(X_train_fold, y_train_fold)

    val_proba = pipeline_fold.predict_proba(X_val_fold)
    oof_preds.iloc[val_idx] = val_proba
    test_preds += pipeline_fold.predict_proba(X_test)

    # Per-fold quick score
    fold_ap = average_precision_score(
        pd.get_dummies(y_val_fold).reindex(columns=classes, fill_value=0),
        val_proba, average='macro'
    )
    print(f"  Fold {i+1}/{N_SPLITS} — train: {len(train_idx)}, val: {len(val_idx)}, val mAP: {fold_ap:.4f}")

test_preds /= N_SPLITS

## 7. Evaluation

In [ ]:
needed_columns = [
    "Clutter", "Cormorants", "Pigeons", "Ducks", "Geese",
    "Gulls", "Birds of Prey", "Waders", "Songbirds",
]

# Build ground-truth OOF solution
solution_df = (
    train_df
    .reset_index()
    .groupby(["track_id", "bird_group"])
    .size()
    .unstack(fill_value=0)
)

# Align OOF predictions to solution_df
oof_aligned = oof_preds.loc[solution_df.index, solution_df.columns]

overall_map = average_precision_score(
    solution_df[needed_columns],
    oof_aligned[needed_columns],
    average='macro'
)

print(f"{'='*50}")
print(f" OOF Macro-Averaged AP (mAP): {overall_map:.4f}")
print(f"{'='*50}")
print("\n Per-Class Average Precision:")
for cls in needed_columns:
    if cls in solution_df.columns and cls in oof_aligned.columns:
        ap = average_precision_score(solution_df[cls], oof_aligned[cls])
        print(f"   {cls:20s}: {ap:.4f}")

## 8. Generate Submission

In [ ]:
submission_df = pd.DataFrame(
    test_preds,
    index=X_test.index,
    columns=classes
)
submission_df.index.name = 'track_id'
submission_df.to_csv('submission.csv')
print(f"Saved submission.csv ({len(submission_df)} rows)")